# 📘 제너레이터와 재귀

**제너레이터**는 값을 한 번에 하나씩 생성하는 이터레이터이고,
**재귀**는 함수가 자기 자신을 호출하는 기법입니다.

**학습 목표:**
- yield로 제너레이터 만들기
- 제너레이터 표현식과 메모리 효율
- 재귀 함수의 기본 구조 (기저 조건 + 재귀 호출)
- 재귀 vs 반복 비교

## 1. 제너레이터 기본

`yield` 키워드로 값을 하나씩 생성합니다. **필요할 때만** 값을 생성하므로 메모리 효율적입니다.

In [ ]:
# ┌───────────────────────────────────────────────┐
# │  제너레이터 vs 리스트                               │
# │  리스트: 모든 값을 메모리에 올림                      │
# │  제너레이터: 필요할 때 하나씩 생성 (yield)           │
# │  대용량 데이터 처리에 유리                            │
# └───────────────────────────────────────────────┘

# 기본 제너레이터
def count_up_to(n):
    """1부터 n까지의 정수를 생성하는 제너레이터"""
    num = 1
    while num <= n:
        yield num
        num += 1
# 제너레이터 객체 생성
gen = count_up_to(5)
print(f"타입: {type(gen)}")    # <class 'generator'>


In [ ]:
# for문으로 순회
print("for문 순회:", end=" ")
for num in count_up_to(5):
    print(num, end=" ")        # 1 2 3 4 5
print()


In [ ]:
# next()로 하나씩 값 얻기
gen = count_up_to(3)
print(f"next(): {next(gen)}")   # 1
print(f"next(): {next(gen)}")    # 2
print(f"next(): {next(gen)}")    # 3
# next(gen)  # StopIteration 예외!
# 리스트로 변환
gen = count_up_to(5)
print(f"list(gen): {list(gen)}")    # [1, 2, 3, 4, 5]


In [ ]:
# ┌─────────────────────────────────────────┐
# │  제너레이터는 한 번만 순회 가능!          │
# │  소진된 제너레이터는 빈 시퀀스와 같음      │
# │  다시 순회하려면 새로 생성해야 함           │
# └─────────────────────────────────────────┘

gen = count_up_to(3)
print(f"첫 순회: {list(gen)}")    # [1, 2, 3]
print(f"두 번째: {list(gen)}")     # [] (소진됨!)


In [ ]:
# 메모리 비교
import sys
list_comp = [x ** 2 for x in range(1000)]
gen_expr = (x ** 2 for x in range(1000))
print(f"\n리스트 크기: {sys.getsizeof(list_comp)} bytes")
print(f"제너레이터 크기: {sys.getsizeof(gen_expr)} bytes")


## 2. 제너레이터 실용 예시

무한 시퀀스, 파이프라인, 대용량 파일 처리에 제너레이터가 유용합니다.

In [ ]:
# 무한 시퀀스 생성
def fibonacci_gen():
    """무한 피보나치 수열 제너레이터"""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


In [ ]:
# itertools.islice로 무한 제너레이터에서 N개만 가져오기
from itertools import islice

fib10 = list(islice(fibonacci_gen(), 10))
print(f"피보나치 10개: {fib10}")


In [ ]:
# 제너레이터 체이닝 (파이프라인)
def read_numbers():
    """숫자 생성"""
    yield from range(1, 11)

def filter_even(nums):
    """짝수만 필터"""
    for n in nums:
        if n % 2 == 0:
            yield n

def square(nums):
    """제곱 계산"""
    for n in nums:
        yield n ** 2
# 파이프라인: 숫자 → 짝수 필터 → 제곱
pipeline = square(filter_even(read_numbers()))
print(f"파이프라인 결과: {list(pipeline)}")


In [ ]:
# yield from: 다른 제너레이터에 위임
def chain(*iterables):
    """여러 이터러블을 연결"""
    for iterable in iterables:
        yield from iterable

result = list(chain([1, 2], [3, 4], [5, 6]))
print(f"연결 결과: {result}")   # [1, 2, 3, 4, 5, 6]


In [ ]:
# 대용량 데이터 처리 (메모리 절약)
def process_large_data(n):
    """1부터 n까지의 제곱을 제너레이터로 생성"""
    for i in range(1, n + 1):
        yield i ** 2

total = sum(process_large_data(1_000_000))
print(f"1~1,000,000 제곱의 합: {total}")


## 3. 재귀 함수

재귀 함수는 **자기 자신을 호출**하는 함수입니다.
반드시 **기저 조건(base case)**이 있어야 무한 재귀를 방지합니다.

```
재귀 함수 구조:
  if 기저 조건:
      return 기본값
  else:
      return 재귀 호출
```

In [ ]:
# ┌─────────────────────────────────────────┐
# │  재귀 함수의 두 가지 필수 요소              │
# │  1. 기저 조건 (base case): 재귀 종료 조건   │
# │  2. 재귀 호출: 더 작은 문제로 분해           │
# │  기저 조건이 없으면 무한 재귀 → RecursionError │
# └─────────────────────────────────────────┘

# 팩토리얼: n! = n × (n-1)!
def factorial(n):
    if n <= 1:         # 기저 조건
        return 1
    return n * factorial(n - 1)    # 재귀 호출

print(f"0! = {factorial(0)}")    # 1
print(f"1! = {factorial(1)}")    # 1
print(f"5! = {factorial(5)}")    # 120
print(f"10! = {factorial(10)}")  # 3628800


In [ ]:
# 피보나치 수열: F(n) = F(n-1) + F(n-2)
def fibonacci(n):
    if n <= 0:
        return 0
    if n == 1:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)

print(f"\n피보나치:")
for i in range(11):
    print(f"F({i}) = {fibonacci(i)}", end="  ")
print()


In [ ]:
# ┌─────────────────────────────────────────────┐
# │  재귀 vs 반복 비교                              │
# │  재귀: 코드가 직관적, 오버헤드 큼              │
# │  반복: 메모리 효율적, 코드가 길어질 수 있음      │
# │  파이썬은 재귀 깊이 제한이 있음 (기본 1000)      │
# └─────────────────────────────────────────────┘

# 반복 버전 팩토리얼 (재귀보다 효율적)
def factorial_iterative(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

print(f"\n재귀 factorial(10) = {factorial(10)}")
print(f"반복 factorial(10) = {factorial_iterative(10)}")
# 재귀 깊이 제한 확인
import sys
print(f"재귀 깊이 제한: {sys.getrecursionlimit()}")


## 🎯 연습 문제

1. 제너레이터 `even_numbers(n)`을 작성하세요. 2부터 n까지의 짝수를 생성합니다.
2. 재귀 함수를 사용해 문자열을 뒤집는 `reverse_string(s)` 함수를 작성하세요.
3. 제너레이터로 피보나치 수열의 첫 20개 값을 생성하고 리스트로 만드세요.
4. 재귀 함수를 사용해 중첩 리스트 `[1, [2, [3, 4]], 5]`의 모든 요소 합을 구하세요.

In [ ]:
# 연습 문제 풀이
